In [ ]:
# Cell 1.5: ADD ALL MISSING IMPORTS
from sklearn.metrics import f1_score
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import matthews_corrcoef
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score, RepeatedStratifiedKFold
from sklearn.model_selection import GridSearchCV
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingClassifier
from xgboost import XGBClassifier
from sklearn.neural_network import MLPClassifier
import seaborn as sns
from sklearn.utils import resample
import warnings
warnings.filterwarnings('ignore')

print("✅ All missing imports added!")
print(f"f1_score imported: {f1_score is not None}")

(2260701, 151)

In [ ]:
import pandas as pd
df=pd.read_csv("/content/drive/MyDrive/Research/loan_data.csv", low_memory=False)

In [ ]:
# Cell 2: Define all 8 models with corrected imports
models = {
    'LDA': {
        'model': LinearDiscriminantAnalysis(),
        'params': {
            'solver': ['svd', 'lsqr', 'eigen'],
            'shrinkage': [None, 'auto', 0.1, 0.5, 0.9]
        }
    },
    'Logistic Regression': {
        'model': LogisticRegression(random_state=42, max_iter=1000, class_weight='balanced'),
        'params': {
            'C': [0.01, 0.1, 1, 10, 100],
            'penalty': ['l1', 'l2', 'elasticnet'],
            'solver': ['saga']
        }
    },
    'Decision Tree': {
        'model': DecisionTreeClassifier(random_state=42, class_weight='balanced'),
        'params': {
            'max_depth': [3, 5, 10, 15, 20, None],
            'min_samples_split': [2, 5, 10, 20],
            'min_samples_leaf': [1, 2, 4, 8],
            'criterion': ['gini', 'entropy']
        }
    },
    'SVM': {
        'model': SVC(random_state=42, class_weight='balanced', probability=True),
        'params': {
            'C': [0.1, 1, 10, 100],
            'gamma': ['scale', 'auto', 0.01, 0.1, 1],
            'kernel': ['rbf', 'poly', 'sigmoid']
        }
    },
    'Random Forest': {
        'model': RandomForestClassifier(random_state=42, class_weight='balanced', n_jobs=-1),
        'params': {
            'n_estimators': [100, 200],
            'max_depth': [10, 20, None],
            'min_samples_split': [2, 5, 10],
            'min_samples_leaf': [1, 2, 4]
        }
    },
    'Gradient Boosting': {
        'model': GradientBoostingClassifier(random_state=42),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0]
        }
    },
    'XGBoost': {
        'model': XGBClassifier(random_state=42, eval_metric='logloss', use_label_encoder=False),
        'params': {
            'n_estimators': [100, 200],
            'learning_rate': [0.01, 0.05, 0.1],
            'max_depth': [3, 5, 7],
            'subsample': [0.8, 0.9, 1.0],
            'colsample_bytree': [0.8, 0.9, 1.0]
        }
    },
    'Neural Network': {
        'model': MLPClassifier(random_state=42, max_iter=500, early_stopping=True),
        'params': {
            'hidden_layer_sizes': [(50,), (100,), (50, 25)],
            'activation': ['relu', 'tanh'],
            'alpha': [0.0001, 0.001],
            'batch_size': [64, 128]
        }
    }
}

print(f"✅ Defined {len(models)} models for benchmarking")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2260701 entries, 0 to 2260700
Columns: 151 entries, id to settlement_term
dtypes: float64(113), object(38)
memory usage: 2.5+ GB


In [ ]:
# Cell 3: Evaluation function with all metrics
def evaluate_model(y_true, y_pred, y_prob=None):
    """
    Comprehensive evaluation with all metrics from the paper
    """
    results = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }

    if y_prob is not None:
        results['roc_auc'] = roc_auc_score(y_true, y_prob)

        # Calculate precision at recall >= 0.65 (matching paper's best recall)
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
        valid_indices = np.where(recalls[:-1] >= 0.65)[0]
        if len(valid_indices) > 0:
            best_idx = valid_indices[np.argmax(precisions[valid_indices])]
            results['precision_at_recall_0.65'] = precisions[best_idx]
        else:
            results['precision_at_recall_0.65'] = 0

    return results

In [ ]:
# Cell 3: Evaluation function with all metrics
def evaluate_model(y_true, y_pred, y_prob=None):
    """
    Comprehensive evaluation with all metrics from the paper
    """
    results = {
        'accuracy': accuracy_score(y_true, y_pred),
        'precision': precision_score(y_true, y_pred, zero_division=0),
        'recall': recall_score(y_true, y_pred, zero_division=0),
        'f1': f1_score(y_true, y_pred, zero_division=0),
        'mcc': matthews_corrcoef(y_true, y_pred),
    }

    if y_prob is not None:
        results['roc_auc'] = roc_auc_score(y_true, y_prob)

        # Calculate precision at recall >= 0.65 (matching paper's best recall)
        precisions, recalls, thresholds = precision_recall_curve(y_true, y_prob)
        valid_indices = np.where(recalls[:-1] >= 0.65)[0]
        if len(valid_indices) > 0:
            best_idx = valid_indices[np.argmax(precisions[valid_indices])]
            results['precision_at_recall_0.65'] = precisions[best_idx]
        else:
            results['precision_at_recall_0.65'] = 0

    return results

In [ ]:
# Cell 4: Simplified benchmarking function (faster execution)
print("🚀 Starting 8-model benchmark...")
print(f"Training data: {X_train_scaled.shape}")
print(f"Test data: {X_test_scaled.shape}")
print("Note: Using simplified grid search for speed\n")

results = []
best_models = {}

for model_name, model_info in models.items():
    print(f"\n📊 Training: {model_name}")
    print("-" * 40)

    try:
        # Use 2-fold CV for speed (you can change to 3 if you have time)
        grid_search = GridSearchCV(
            model_info['model'],
            model_info['params'],
            cv=2,  # Reduced from 3 to 2 for speed
            scoring='roc_auc',
            n_jobs=-1,
            verbose=0
        )

        # Train
        grid_search.fit(X_train_scaled, y_train)
        best_model = grid_search.best_estimator_
        best_models[model_name] = best_model

        # Predict
        y_pred = best_model.predict(X_test_scaled)

        # Get probabilities (handle models without predict_proba)
        if hasattr(best_model, 'predict_proba'):
            y_prob = best_model.predict_proba(X_test_scaled)[:, 1]
        else:
            y_prob = None

        # Evaluate
        metrics = evaluate_model(y_test, y_pred, y_prob)
        metrics['model'] = model_name
        metrics['best_params'] = str(grid_search.best_params_)
        metrics['cv_score'] = grid_search.best_score_
        results.append(metrics)

        # Print results
        print(f"  ✓ CV Score: {grid_search.best_score_:.4f}")
        print(f"  ✓ ROC-AUC: {metrics.get('roc_auc', 0):.4f}")
        print(f"  ✓ Recall: {metrics['recall']:.4f}")
        print(f"  ✓ Precision: {metrics['precision']:.4f}")
        if 'precision_at_recall_0.65' in metrics:
            print(f"  ✓ Trigger Precision (recall≥0.65): {metrics['precision_at_recall_0.65']:.4f}")

    except Exception as e:
        print(f"  ✗ Error: {str(e)}")
        continue

# Create results dataframe
if results:
    results_df = pd.DataFrame(results)
    print("\n" + "="*60)
    print("✅ BENCHMARKING COMPLETE")
    print("="*60)

    # Sort by ROC-AUC
    results_df = results_df.sort_values('roc_auc', ascending=False)
    display(results_df[['model', 'roc_auc', 'recall', 'precision', 'precision_at_recall_0.65', 'cv_score']].round(4))
else:
    print("❌ No models completed successfully")

In [ ]:
# Cell 5: Find and analyze best model
if results:
    best_model_row = results_df.loc[results_df['roc_auc'].idxmax()]
    best_model_name = best_model_row['model']

    print("\n" + "="*60)
    print(f"🏆 BEST MODEL: {best_model_name}")
    print("="*60)
    print(f"   ROC-AUC: {best_model_row['roc_auc']:.4f}")
    print(f"   Recall: {best_model_row['recall']:.4f}")
    print(f"   Precision: {best_model_row['precision']:.4f}")
    print(f"   Trigger Precision: {best_model_row.get('precision_at_recall_0.65', 0):.4f}")
    print(f"   CV Score: {best_model_row['cv_score']:.4f}")

    # Compare with Random Forest
    rf_row = results_df[results_df['model'] == 'Random Forest']
    if len(rf_row) > 0:
        rf_auc = rf_row['roc_auc'].values[0]
        best_auc = best_model_row['roc_auc']

        print("\n📊 Comparison with Random Forest:")
        print("-" * 40)
        if best_model_name == 'Random Forest':
            print("✅ Random Forest remains the best model!")
        else:
            improvement = ((best_auc - rf_auc) / rf_auc) * 100
            print(f"📈 {best_model_name} outperforms Random Forest by {improvement:.2f}%")
            print(f"   Random Forest ROC-AUC: {rf_auc:.4f}")
            print(f"   Best Model ROC-AUC: {best_auc:.4f}")

    # Show full rankings
    print("\n📋 Complete Rankings:")
    print("-" * 40)
    for idx, row in results_df.iterrows():
        medal = "🥇" if idx == 0 else "🥈" if idx == 1 else "🥉" if idx == 2 else "  "
        print(f"{medal} {row['model']}: ROC-AUC={row['roc_auc']:.4f}, Recall={row['recall']:.4f}")

In [ ]:
# Cell 6: Compare with Elsevier paper
print("\n" + "="*60)
print("📋 COMPARISON WITH ELSEVIER PAPER")
print("="*60)

comparison = pd.DataFrame({
    'Metric': [
        'Best Model',
        'Dataset',
        'Sample Size',
        'Default Rate',
        'Recall (Best Model)',
        'Trigger Precision',
        'Models Tested',
        'Interpretability'
    ],
    'Elsevier Paper': [
        'Random Forest',
        'Dutch Bank (Commercial)',
        '548,210 observations',
        '0.8%',
        '0.65-0.68',
        '0.12-0.15',
        '8',
        'SHAP'
    ],
    'Your Results': [
        best_model_name if results else 'N/A',
        'LendingClub (Consumer)',
        f"{len(df):,}",
        f"{(y.sum()/len(y)*100):.1f}%",
        f"{best_model_row['recall']:.3f}" if results else 'N/A',
        f"{best_model_row.get('precision_at_recall_0.65', 0):.3f}" if results else 'N/A',
        str(len(results)),
        'Feature Importance'
    ]
})
display(comparison)

# Highlight improvements
if results:
    print("\n✨ KEY FINDINGS:")
    print("-" * 40)

    if best_model_row.get('precision_at_recall_0.65', 0) > 0.15:
        print(f"✅ Your trigger precision ({best_model_row['precision_at_recall_0.65']:.3f}) beats the paper's (0.15)!")
    if best_model_row['recall'] > 0.68:
        print(f"✅ Your recall ({best_model_row['recall']:.3f}) beats the paper's best (0.68)!")
    if best_model_name != 'Random Forest':
        print(f"✅ You found a better model than Random Forest: {best_model_name}")
    if len(results) == 8:
        print("✅ You successfully benchmarked all 8 models!")

In [ ]:
# Cell 7: Visualize results
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Plot 1: ROC-AUC comparison
ax1 = axes[0, 0]
sorted_df = results_df.sort_values('roc_auc', ascending=True)
colors = ['gold' if m == best_model_name else 'skyblue' for m in sorted_df['model']]
bars = ax1.barh(sorted_df['model'], sorted_df['roc_auc'], color=colors)
ax1.set_xlabel('ROC-AUC Score')
ax1.set_title('Model Performance Comparison')
ax1.set_xlim(0.5, 1.0)
ax1.axvline(x=0.65, color='red', linestyle='--', alpha=0.5, label='Paper Recall Threshold')
ax1.legend()

# Plot 2: Precision-Recall tradeoff
ax2 = axes[0, 1]
scatter = ax2.scatter(results_df['recall'], results_df['precision'],
                     c=results_df['roc_auc'], cmap='viridis', s=200)
for _, row in results_df.iterrows():
    ax2.annotate(row['model'], (row['recall'] + 0.01, row['precision'] - 0.01), fontsize=9)
ax2.set_xlabel('Recall')
ax2.set_ylabel('Precision')
ax2.set_title('Precision-Recall Tradeoff')
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)
plt.colorbar(scatter, ax=ax2, label='ROC-AUC')

# Plot 3: Trigger Precision Comparison
ax3 = axes[1, 0]
trigger_models = results_df.nlargest(5, 'precision_at_recall_0.65')
ax3.bar(trigger_models['model'], trigger_models['precision_at_recall_0.65'], color='lightcoral')
ax3.set_xlabel('Model')
ax3.set_ylabel('Precision at Recall ≥ 0.65')
ax3.set_title('Trigger Precision (Watchlist Metric)')
ax3.tick_params(axis='x', rotation=45)
ax3.axhline(y=0.15, color='red', linestyle='--', label='Paper Baseline (0.15)')
ax3.legend()

# Plot 4: Paper Comparison
ax4 = axes[1, 1]
metrics = ['Recall', 'Trigger Precision']
paper_values = [0.665, 0.135]  # Midpoint of paper's ranges
your_values = [best_model_row['recall'], best_model_row.get('precision_at_recall_0.65', 0)]

x = np.arange(len(metrics))
width = 0.35
ax4.bar(x - width/2, paper_values, width, label='Elsevier Paper', color='lightgray')
ax4.bar(x + width/2, your_values, width, label='Your Results', color='lightgreen')
ax4.set_ylabel('Score')
ax4.set_title('Comparison with Paper')
ax4.set_xticks(x)
ax4.set_xticklabels(metrics)
ax4.legend()
ax4.set_ylim(0, 1)

plt.tight_layout()
plt.show()

In [ ]:
# Cell 8: Save best model
if results:
    # Save the best model
    best_model = best_models[best_model_name]

    import joblib
    import os

    model_dir = "/content/drive/MyDrive/Research/model"
    os.makedirs(model_dir, exist_ok=True)

    # Save with descriptive name
    joblib.dump(best_model, os.path.join(model_dir, f"best_model_{best_model_name.replace(' ', '_')}.pkl"))
    joblib.dump(X.columns.tolist(), os.path.join(model_dir, "feature_columns.pkl"))

    # Save results
    results_df.to_csv(os.path.join(model_dir, "benchmark_results.csv"), index=False)

    print(f"\n✅ Best model saved: {best_model_name}")
    print(f"📊 Results saved to: {model_dir}")